# Camada Gold — modelo dimensional com Spark SQL

A fato tem uma linha por `source_row_id` da Prata (uma linha por ocorrência do snapshot atual), não por código de emenda. As SKs dimensionais são hashes SHA-256 determinísticos dos atributos naturais; cada dimensão contém o membro `UNKNOWN` para fatos sem correspondência. As tabelas são reconstruídas como snapshots Delta no catálogo/schema atual.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Dimensão de tempo
spark.sql("""
CREATE OR REPLACE TABLE emendas_parlamentares_dim_tempo
USING DELTA AS
SELECT
    'UNKNOWN' AS sk_tempo,
    CAST(NULL AS INT) AS ano_emenda
UNION ALL
SELECT
    sha2(to_json(named_struct('ano_emenda', ano_emenda)), 256) AS sk_tempo,
    ano_emenda
FROM (
    SELECT DISTINCT ano_emenda
    FROM emendas_parlamentares_prata
    WHERE ano_emenda IS NOT NULL
) AS origem
""")

# Dimensão de autores
spark.sql("""
CREATE OR REPLACE TABLE emendas_parlamentares_dim_autor
USING DELTA AS
SELECT
    'UNKNOWN' AS sk_autor,
    CAST(NULL AS STRING) AS codigo_autor,
    CAST(NULL AS STRING) AS nome_autor
UNION ALL
SELECT
    sha2(to_json(named_struct(
        'codigo_autor', codigo_autor,
        'nome_autor', nome_autor
    )), 256) AS sk_autor,
    codigo_autor,
    nome_autor
FROM (
    SELECT DISTINCT codigo_autor, nome_autor
    FROM emendas_parlamentares_prata
    WHERE codigo_autor IS NOT NULL OR nome_autor IS NOT NULL
) AS origem
""")

# Dimensão de localidade
spark.sql("""
CREATE OR REPLACE TABLE emendas_parlamentares_dim_localidade
USING DELTA AS
SELECT
    'UNKNOWN' AS sk_localidade,
    CAST(NULL AS STRING) AS codigo_municipio_ibge,
    CAST(NULL AS STRING) AS municipio,
    CAST(NULL AS INT) AS codigo_uf_ibge,
    CAST(NULL AS STRING) AS uf,
    CAST(NULL AS STRING) AS regiao,
    CAST(NULL AS STRING) AS localidade_aplicacao
UNION ALL
SELECT
    sha2(to_json(named_struct(
        'codigo_municipio_ibge', codigo_municipio_ibge,
        'municipio', municipio,
        'codigo_uf_ibge', codigo_uf_ibge,
        'uf', uf,
        'regiao', regiao,
        'localidade_aplicacao', localidade_aplicacao
    )), 256) AS sk_localidade,
    codigo_municipio_ibge,
    municipio,
    codigo_uf_ibge,
    uf,
    regiao,
    localidade_aplicacao
FROM (
    SELECT DISTINCT
        codigo_municipio_ibge,
        municipio,
        codigo_uf_ibge,
        uf,
        regiao,
        localidade_aplicacao
    FROM emendas_parlamentares_prata
    WHERE codigo_municipio_ibge IS NOT NULL
       OR municipio IS NOT NULL
       OR codigo_uf_ibge IS NOT NULL
       OR uf IS NOT NULL
       OR regiao IS NOT NULL
       OR localidade_aplicacao IS NOT NULL
) AS origem
""")

# Dimensão de classificação/programa orçamentário
spark.sql("""
CREATE OR REPLACE TABLE emendas_parlamentares_dim_programa_orcamentario
USING DELTA AS
SELECT
    'UNKNOWN' AS sk_programa_orcamentario,
    CAST(NULL AS STRING) AS codigo_funcao,
    CAST(NULL AS STRING) AS nome_funcao,
    CAST(NULL AS STRING) AS codigo_subfuncao,
    CAST(NULL AS STRING) AS nome_subfuncao,
    CAST(NULL AS STRING) AS codigo_programa,
    CAST(NULL AS STRING) AS nome_programa,
    CAST(NULL AS STRING) AS codigo_acao,
    CAST(NULL AS STRING) AS nome_acao,
    CAST(NULL AS STRING) AS codigo_plano_orcamentario,
    CAST(NULL AS STRING) AS nome_plano_orcamentario
UNION ALL
SELECT
    sha2(to_json(named_struct(
        'codigo_funcao', codigo_funcao,
        'nome_funcao', nome_funcao,
        'codigo_subfuncao', codigo_subfuncao,
        'nome_subfuncao', nome_subfuncao,
        'codigo_programa', codigo_programa,
        'nome_programa', nome_programa,
        'codigo_acao', codigo_acao,
        'nome_acao', nome_acao,
        'codigo_plano_orcamentario', codigo_plano_orcamentario,
        'nome_plano_orcamentario', nome_plano_orcamentario
    )), 256) AS sk_programa_orcamentario,
    codigo_funcao,
    nome_funcao,
    codigo_subfuncao,
    nome_subfuncao,
    codigo_programa,
    nome_programa,
    codigo_acao,
    nome_acao,
    codigo_plano_orcamentario,
    nome_plano_orcamentario
FROM (
    SELECT DISTINCT
        codigo_funcao,
        nome_funcao,
        codigo_subfuncao,
        nome_subfuncao,
        codigo_programa,
        nome_programa,
        codigo_acao,
        nome_acao,
        codigo_plano_orcamentario,
        nome_plano_orcamentario
    FROM emendas_parlamentares_prata
    WHERE codigo_funcao IS NOT NULL
       OR nome_funcao IS NOT NULL
       OR codigo_subfuncao IS NOT NULL
       OR nome_subfuncao IS NOT NULL
       OR codigo_programa IS NOT NULL
       OR nome_programa IS NOT NULL
       OR codigo_acao IS NOT NULL
       OR nome_acao IS NOT NULL
       OR codigo_plano_orcamentario IS NOT NULL
       OR nome_plano_orcamentario IS NOT NULL
) AS origem
""")

In [ ]:
# A chave da fato vem da linha de origem, não do código (repetível) da emenda.
spark.sql("""
CREATE OR REPLACE TABLE emendas_parlamentares_fato
USING DELTA AS
SELECT
    e.source_row_id AS id_fato_emenda,
    e.source_batch_id,
    e.source_file,
    coalesce(t.sk_tempo, 'UNKNOWN') AS sk_tempo,
    coalesce(a.sk_autor, 'UNKNOWN') AS sk_autor,
    coalesce(l.sk_localidade, 'UNKNOWN') AS sk_localidade,
    coalesce(p.sk_programa_orcamentario, 'UNKNOWN') AS sk_programa_orcamentario,
    e.codigo_emenda,
    e.numero_emenda,
    e.tipo_emenda,
    e.valor_empenhado,
    e.valor_liquidado,
    e.valor_pago,
    e.valor_rap_inscritos,
    e.valor_rap_cancelados,
    e.valor_rap_pagos,
    e.data_carga
FROM emendas_parlamentares_prata AS e
LEFT JOIN emendas_parlamentares_dim_tempo AS t
    ON e.ano_emenda <=> t.ano_emenda
LEFT JOIN emendas_parlamentares_dim_autor AS a
    ON e.codigo_autor <=> a.codigo_autor
   AND e.nome_autor <=> a.nome_autor
LEFT JOIN emendas_parlamentares_dim_localidade AS l
    ON e.codigo_municipio_ibge <=> l.codigo_municipio_ibge
   AND e.municipio <=> l.municipio
   AND e.codigo_uf_ibge <=> l.codigo_uf_ibge
   AND e.uf <=> l.uf
   AND e.regiao <=> l.regiao
   AND e.localidade_aplicacao <=> l.localidade_aplicacao
LEFT JOIN emendas_parlamentares_dim_programa_orcamentario AS p
    ON e.codigo_funcao <=> p.codigo_funcao
   AND e.nome_funcao <=> p.nome_funcao
   AND e.codigo_subfuncao <=> p.codigo_subfuncao
   AND e.nome_subfuncao <=> p.nome_subfuncao
   AND e.codigo_programa <=> p.codigo_programa
   AND e.nome_programa <=> p.nome_programa
   AND e.codigo_acao <=> p.codigo_acao
   AND e.nome_acao <=> p.nome_acao
   AND e.codigo_plano_orcamentario <=> p.codigo_plano_orcamentario
   AND e.nome_plano_orcamentario <=> p.nome_plano_orcamentario
""")

In [ ]:
checks = spark.sql("""
SELECT 'silver_fact_row_count' AS check_name,
       (SELECT COUNT(*) FROM emendas_parlamentares_prata) AS expected_count,
       (SELECT COUNT(*) FROM emendas_parlamentares_fato) AS actual_count
UNION ALL
SELECT 'fact_null_or_duplicate_source_id',
       CAST(0 AS BIGINT),
       (SELECT COUNT(*) FROM emendas_parlamentares_fato WHERE id_fato_emenda IS NULL)
       + (SELECT COUNT(*) FROM (
            SELECT id_fato_emenda FROM emendas_parlamentares_fato
            GROUP BY id_fato_emenda HAVING COUNT(*) > 1
          ) AS duplicados)
UNION ALL
SELECT 'fact_foreign_keys_not_null',
       CAST(0 AS BIGINT),
       (SELECT COUNT(*) FROM emendas_parlamentares_fato
        WHERE sk_tempo IS NULL OR sk_autor IS NULL
           OR sk_localidade IS NULL OR sk_programa_orcamentario IS NULL)
UNION ALL
SELECT 'invalid_conversions',
       CAST(0 AS BIGINT),
       (SELECT COUNT(*) FROM emendas_parlamentares_rejeicoes)
""").collect()

for check in checks:
    print(f"{check['check_name']}: esperado={check['expected_count']}, obtido={check['actual_count']}")
    if check["expected_count"] != check["actual_count"]:
        raise ValueError(f"Falha na validação Gold: {check['check_name']}")

spark.sql("""
SELECT
    l.uf AS estado,
    COUNT(*) AS total_registros,
    SUM(f.valor_empenhado) AS total_empenhado,
    SUM(f.valor_pago) AS total_pago
FROM emendas_parlamentares_fato AS f
JOIN emendas_parlamentares_dim_localidade AS l
    ON f.sk_localidade = l.sk_localidade
GROUP BY l.uf
ORDER BY total_registros DESC
""").show(20, truncate=False)